# UNIT400-TWIN · M3 — catalyst screening

**Question M3 answers:** for a given catalyst, *where* does R‑400 make on‑spec product, where does it lose conversion, and where does the hot spot run away — as a function of what the operator can move: **jacket inlet temperature (04TIC411 SP)** and **feed rate**.

Basis = `basis.json` (your plant, private) or `basis.example.json`: jacket circulation, feed preheat, H₂ in ratio to feed, 04TIC410 AUTO.

| result | meaning |
|---|---|
| **OK** | 04AT400 ≤ 5 % unreacted and peak hot spot ≤ 200 °C |
| **LOW** | conversion lost (AT400 > 5 %) |
| **HOT** | hot spot exceeded 200 °C at any time (runaway risk) |

The 5 % and 200 °C are screening assumptions (edit `S.SPEC_AT400`, `S.HOT_LIMIT`), not plant trip settings.
Each grid point = heat the bed at that jacket SP, then 40 min of reaction. `QUICK = True` ≈ 10 min; `False` ≈ 25 min.

In [ ]:
import sys, os, csv, time
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'twin'))
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import catalyst as C, screening as S
QUICK = True
lib = C.load_library(os.path.join('..', 'twin', 'catalysts.json')); base = lib['mimic_baseline']
SPS   = [115, 125, 135, 145] if QUICK else [115, 125, 135, 145, 155]
ACTS  = [60, 80, 100, 120] if QUICK else [50, 60, 70, 80, 90, 100, 110, 120]
FEEDS = [int(S.FEED * k) for k in ((0.7, 1.0, 1.25) if QUICK else (0.6, 0.8, 1.0, 1.25, 1.45))]
CODE = {'LOW': 0, 'OK': 1, 'HOT': 2}
cmap = ListedColormap(['#f2c14e', '#5fbf8a', '#e05a5a'])

def grid(rowvals, make):
    return [[make(rv, sp) for sp in SPS] for rv in rowvals]

def show(G, rowvals, ylabel, title):
    fig, ax = plt.subplots(figsize=(1.6 * len(SPS) + 2, 0.7 * len(rowvals) + 1.6))
    ax.imshow([[CODE[r['result']] for r in row] for row in G], cmap=cmap, vmin=0, vmax=2, aspect='auto')
    for i, row in enumerate(G):
        for j, r in enumerate(row):
            txt = '%s\n%.0f %%' % (r['result'], r['at400']) if r['result'] != 'HOT' else 'HOT\n%.0f °C' % r['peak']
            ax.text(j, i, txt, ha='center', va='center', fontsize=8)
    ax.set_xticks(range(len(SPS))); ax.set_xticklabels(['%d °C' % s for s in SPS]); ax.set_xlabel('jacket inlet SP (04TIC411)')
    ax.set_yticks(range(len(rowvals))); ax.set_yticklabels(rowvals); ax.set_ylabel(ylabel); ax.set_title(title)
    plt.tight_layout(); plt.show()
print('ready')

## A — how active must the catalyst be? (basis feed)

In [ ]:
t0 = time.time()
GA = grid(ACTS, lambda a, sp: S.screen_point(C.CatalystProfile('act%d' % a, activity_pct=a), sp))
show(GA, ['%d %%' % a for a in ACTS], 'catalyst activity (04CAT400)', 'A  activity × jacket temperature')
print('%.1f min' % ((time.time() - t0) / 60))

**Read it:** a more active catalyst opens the window to a cooler jacket. Too little activity and no jacket setting in range reaches spec — the positive‑feedback loop never catches.

## B — how much feed can the baseline catalyst take?

In [ ]:
GB = grid(FEEDS, lambda f, sp: S.screen_point(base, sp, feed=f))
show(GB, ['%d l/h' % f for f in FEEDS], 'feed 04FIC400', 'B  feed rate × jacket temperature (baseline catalyst)')

**Read it:** more feed = less dwell time = needs a hotter jacket.

## C — catalyst types side by side

In [ ]:
names = [n for n, p in lib.items() if p.kd_per_h == 0]
GC = grid(names, lambda n, sp: S.screen_point(lib[n], sp))
show(GC, names, '', 'C  catalyst type × jacket temperature (basis feed)')

**Read it:** the strongly temperature‑sensitive catalyst (Arrhenius, Ea 60) has **no safe on‑spec window** here: too cool → conversion lost, hot enough → runaway. That is a *reject* (or a different reactor/cooling design), found without touching hardware.

## D — deactivation: how long does a dying catalyst stay on spec?

In [ ]:
KDS, DSP = [0.02, 0.05, 0.1], [125, 135, 145]
H = 4 if QUICK else 8
print('hours on spec (AT400 <= %.0f %%), feed %d l/h, max %d h' % (S.SPEC_AT400, S.FEED, H))
print('%10s' % 'kd per h' + ''.join('%9s' % ('jkt %d' % s) for s in DSP))
for kd in KDS:
    line = '%10s' % kd
    for sp in DSP:
        t_lost, trail = S.time_on_spec(C.CatalystProfile('kd%g' % kd, kd_per_h=kd), sp, hours=H)
        line += '%9s' % ('never' if t_lost is None and not trail else ('>%dh' % H if t_lost is None else '%.1f h' % t_lost))
    print(line)

**Read it:** raising the jacket buys time, but not much. At 5 %/h loss even 145 °C holds spec only ~1.5 h — compensation by temperature has a hard limit; the catalyst has to be replaced or regenerated.

## Take‑aways for the operator / student
- The reactor runs on a **narrow window** of jacket temperature for a given feed.
- **More feed → hotter jacket needed.** Less active catalyst → hotter jacket needed.
- **A fast, temperature‑sensitive catalyst can have no safe window at all** in this reactor.
- **Rising AT400 at constant settings** = catalyst dying; raising the jacket only delays it.